In [0]:
%pip install geopandas shapely --quiet
dbutils.library.restartPython()

In [0]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from datetime import datetime, timezone

In [0]:
CATALOG = "gobierno_abierto"
SCHEMA_SILVER = "b_silver"
SCHEMA_GOLD = "c_gold"

In [0]:
## 1. Lectura de Silver y armado de GeoDataFrames

stops_pdf = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.stops").toPandas()
barrios_pdf = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.barrios").toPandas()

print(f"Paradas (silver): {len(stops_pdf)}")
print(f"Barrios (silver): {len(barrios_pdf)}")

stops_gdf = gpd.GeoDataFrame(
    stops_pdf,
    geometry=stops_pdf["geometry_wkt"].apply(wkt.loads),
    crs="EPSG:4326",
)

barrios_gdf = gpd.GeoDataFrame(
    barrios_pdf,
    geometry=barrios_pdf["geometry_wkt"].apply(wkt.loads),
    crs="EPSG:4326",
)

print(f"stops_gdf: {len(stops_gdf)} filas, CRS={stops_gdf.crs}")
print(f"barrios_gdf: {len(barrios_gdf)} filas, CRS={barrios_gdf.crs}")

In [0]:
## 2. Join espacial (point-in-polygon)

# 1. Buffer chico SOLO para cerrar los micro-gaps de digitalización (~5m)
TOLERANCIA_GRADOS = 0.00005  # ~5m

barrios_gdf_buffer = barrios_gdf.copy()
barrios_gdf_buffer["geometry"] = barrios_gdf_buffer.geometry.buffer(TOLERANCIA_GRADOS)

join_result = gpd.sjoin(
    stops_gdf,
    barrios_gdf_buffer[["barrio_id", "nombre", "geometry"]],
    how="left",
    predicate="within",
)
join_result = join_result.drop_duplicates(subset=["stop_id"], keep="first")

sin_barrio_final = join_result["barrio_id"].isna().sum()
print(f"Paradas sin barrio tras buffer de micro-gaps: {sin_barrio_final} de {len(join_result)}")
# Debería bajar de 488 a ~458 (quedan las de cobertura real faltante)

In [0]:
# Paradas sin barrio: ¿están cerca de algún polígono (problema de límite/precisión)
# o están realmente lejos de todos los barrios (fuera del área cubierta por el dataset)?
sin_barrio_gdf = stops_gdf[join_result["barrio_id"].isna()].copy()

# Reintentar con predicate más permisivo: "intersects" en vez de "within"
# (within excluye puntos justo sobre el borde del polígono)
retry = gpd.sjoin(
    sin_barrio_gdf,
    barrios_gdf[["barrio_id", "nombre", "geometry"]],
    how="left",
    predicate="intersects",
)
recuperadas = retry["barrio_id"].notna().sum()
print(f"De las {len(sin_barrio_gdf)} sin barrio, {recuperadas} se recuperan con 'intersects' (estaban justo en el límite)")

# Las que siguen sin match ni con intersects: ver dónde están (¿fuera de la ciudad?)
sigue_sin_barrio = retry[retry["barrio_id"].isna()]
print(f"Siguen sin barrio incluso con intersects: {len(sigue_sin_barrio)}")
display(sigue_sin_barrio[["stop_id", "stop_name", "stop_lat", "stop_lon"]].head(20))

In [0]:
## verificaciones de barrios faltantes

from shapely.ops import unary_union

# Unimos todos los polígonos de barrios en una sola geometría
cobertura_total = unary_union(barrios_gdf.geometry)

# Para cada parada sin barrio, medimos la distancia mínima al polígono más cercano
sigue_sin_barrio = sigue_sin_barrio.copy()
sigue_sin_barrio["distancia_a_barrio_mas_cercano_m"] = sigue_sin_barrio.geometry.apply(
    lambda p: barrios_gdf.geometry.distance(p).min() * 111000  # aprox grados -> metros
)

print(sigue_sin_barrio["distancia_a_barrio_mas_cercano_m"].describe())
display(sigue_sin_barrio.sort_values("distancia_a_barrio_mas_cercano_m")[
    ["stop_id", "stop_name", "distancia_a_barrio_mas_cercano_m"]
].head(20))

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))
barrios_gdf.plot(ax=ax, color="lightgray", edgecolor="gray", linewidth=0.3)
sigue_sin_barrio.plot(ax=ax, color="red", markersize=8)
ax.set_title("Paradas sin barrio asignado (rojo) sobre polígonos de barrios (gris)")
plt.show()

from shapely.ops import unary_union

# Recalcula la distancia de cada parada sin barrio al polígono más cercano
sigue_sin_barrio = sigue_sin_barrio.copy()
sigue_sin_barrio["distancia_aprox_m"] = sigue_sin_barrio.geometry.apply(
    lambda p: barrios_gdf.geometry.distance(p).min() * 111000
)

# Separar en dos grupos según la distancia al barrio más cercano
GAP_MENOR = sigue_sin_barrio[sigue_sin_barrio["distancia_aprox_m"] <= 5]
FUERA_COBERTURA = sigue_sin_barrio[sigue_sin_barrio["distancia_aprox_m"] > 5]

print(f"Micro-gaps entre polígonos (≤5m): {len(GAP_MENOR)}")
print(f"Fuera de toda cobertura (>5m): {len(FUERA_COBERTURA)}")
print(FUERA_COBERTURA["distancia_aprox_m"].describe())

In [0]:
## 3. `gold.stops_by_barrio`

stops_by_barrio = join_result.rename(columns={"nombre": "barrio_nombre"})[
    ["stop_id", "barrio_id", "barrio_nombre"]
].copy()
stops_by_barrio["_created_at"] = datetime.now(timezone.utc)

spark.createDataFrame(stops_by_barrio).write.mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.stops_by_barrio"
)

count = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.stops_by_barrio").count()
print(f"{CATALOG}.{SCHEMA_GOLD}.stops_by_barrio: {count} filas escritas.")
display(spark.table(f"{CATALOG}.{SCHEMA_GOLD}.stops_by_barrio").limit(5))

In [0]:
problematicos = barrio_coverage[
    barrio_coverage["area_km2"].isna()
    | (barrio_coverage["area_km2"] == 0)
    | ~barrio_coverage["paradas_por_km2"].apply(lambda x: pd.notna(x) and x != float("inf"))
]
print(f"Barrios con area_km2 nula/cero o paradas_por_km2 no finito: {len(problematicos)}")
display(problematicos[["barrio_id", "barrio_nombre", "area_km2", "cantidad_paradas", "paradas_por_km2"]])

In [0]:
## 4. `gold.barrio_coverage`

paradas_por_barrio = (
    stops_by_barrio.dropna(subset=["barrio_id"])
    .groupby("barrio_id")
    .size()
    .rename("cantidad_paradas")
)

barrio_coverage = barrios_pdf[["barrio_id", "nombre", "area_km2"]].rename(
    columns={"nombre": "barrio_nombre"}
).merge(paradas_por_barrio, on="barrio_id", how="left")

barrio_coverage["cantidad_paradas"] = barrio_coverage["cantidad_paradas"].fillna(0).astype(int)

# area_km2 nula/cero -> paradas_por_km2 queda como None (división indefinida),
# en vez de generar inf/NaN silenciosamente
barrio_coverage["paradas_por_km2"] = barrio_coverage.apply(
    lambda r: r["cantidad_paradas"] / r["area_km2"] if pd.notna(r["area_km2"]) and r["area_km2"] > 0 else None,
    axis=1,
)

# rank() deja NaN en las filas sin paradas_por_km2 -> Int64 (nullable) en vez de
# int permite guardar esas filas como <NA> sin que falle el cast
barrio_coverage["ranking_cobertura"] = (
    barrio_coverage["paradas_por_km2"].rank(method="min", ascending=False).astype("Int64")
)

barrio_coverage = barrio_coverage.sort_values("ranking_cobertura")
barrio_coverage["_created_at"] = datetime.now(timezone.utc)

# Log explícito de los barrios excluidos del ranking, para que no se pierdan de vista
excluidos = barrio_coverage[barrio_coverage["ranking_cobertura"].isna()]
if len(excluidos) > 0:
    print(f"⚠️ {len(excluidos)} barrio(s) excluidos del ranking por área inválida en la fuente:")
    display(excluidos[["barrio_id", "barrio_nombre", "area_km2", "cantidad_paradas"]])

barrio_coverage.head(10)

spark.createDataFrame(barrio_coverage).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.barrio_coverage"
)

count = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.barrio_coverage").count()
print(f"{CATALOG}.{SCHEMA_GOLD}.barrio_coverage: {count} filas escritas.")